# 20. Citation Accuracy: Do Graph-Based Answers Cite Their Sources Better Than Vector RAG?

> **Outputs in this copy come from a 25-question test run per benchmark (`SMASH_MODE=test`), a smoke test of the
> pipeline, not the confidence run.** The full run (1,000 questions per benchmark) is started with
> `SMASH_MODE=batch`; until then the numbers below must not be quoted.

**Presentation map:** Slide 5 (a graph edge can point to the passage it came from: provenance), Task 3.2
(provenance), Task 1.4 (graph-based ranking of passages).

The tutorial claims that a knowledge graph addresses a limitation of LLMs: the weights cannot cite a source, whereas a
graph edge can point to the passage it was extracted from. Notebooks 01–19 never test that claim. This notebook does,
with the citation measures of the **ALCE** benchmark (Gao et al., EMNLP 2023) on the data and indexes of Notebook 11.

**Research question.** When an LLM must cite the sources of every statement, are the citations of graph-based answers
more accurate than those of vector RAG?

**Data.** MuSiQue (answerable, dev) and 2WikiMultiHopQA (dev), 1,000 questions each over pools of 11,656 and 6,119
Wikipedia passages: the benchmarks, passage pools, embeddings and extracted graph of Notebook 11, reused from its cache
(no index is rebuilt). Both benchmarks label the gold supporting paragraphs of every question.

**Three conditions, one answer prompt:**
- **(a) Vector RAG** — the 5 passages most similar to the question (Notebook 11's baseline), cited as `[1]`–`[5]`.
- **(b) Graph passages** — the 5 passages ranked highest by HippoRAG-style personalised PageRank (Notebook 11's graph
  retriever), cited as `[1]`–`[5]`.
- **(c) Graph facts** — the 15 extracted triples most similar to the question, cited as `[1]`–`[15]`. Each triple keeps
  the id of the passage it came from, so a citation resolves to its **source passage** through provenance: the slide-5
  claim in its most direct form.

Every answer has at most three short sentences, each ending with its sources in square brackets, and a last line
`Answer: <short answer> [n]`. The answer line is scored as the statement *“The answer to ⟨question⟩ is ⟨answer⟩.”*

**Measurement.**
1. **Answer quality:** exact match and token F1 against the gold answer, as in Notebook 11.
2. **Gold-based citation precision and recall (no judge):** the share of cited passages that are gold supporting
   paragraphs, and the share of gold supporting paragraphs that are cited.
3. **ALCE citation recall and precision (LLM judge):** citation recall is the share of statements whose cited passages
   together entail them (a statement without citations counts as unsupported); citation precision is the share of
   (statement, cited passage) pairs in which the passage is relevant to the statement. One structured judge call per
   answer sees the question, the cited passages and the numbered statements.

All differences are paired over questions (graph − vector, facts − vector) with 95% bootstrap intervals.

**Expected outcome.** If provenance makes citations more accurate, (b) and especially (c) should exceed (a) on citation
precision and recall. If citation quality follows what is retrieved, (b) should track its recall gain from Notebook 11
(large on 2Wiki, small on MuSiQue), and (c) may cite precisely but answer less often, because triples carry less text.

**Model:** `gpt-6-luna`, no reasoning, temperature 0, through the OpenAI **Batch API** (helper `smash_batch.py`), for both
the answers and the judge. New calls are cached as the stages `<benchmark>_cited_answers` and
`<benchmark>_citation_judge` in Notebook 11's cache directory; the default `SMASH_MODE=replay` re-runs the saved results
without calling the API.

In [1]:
import json, os, re, string, time, random
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/musique_2wiki"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
N_Q = 25 if TEST else 1000           # questions per benchmark (test mode: the first 25 of the same fixed order)

def load(name):
    qs = json.load(open(f"{BASE}/{name}.json"))
    corpus = json.load(open(f"{BASE}/{name}_corpus.json"))
    return qs, corpus

DATA = {}
for name in ("musique", "2wikimultihopqa"):
    qs, corpus = load(name)
    for q in qs:                                     # one schema for both benchmarks
        if name == "musique":
            q["qid"], q["golds"] = q["id"], [q["answer"]] + q.get("answer_aliases", [])
            q["support"] = {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"] if p["is_supporting"]}
            q["group"] = q["id"].split("hop")[0] + "-hop"
        else:
            q["qid"], q["golds"] = q["_id"], [q["answer"]]
            ctx = {t: " ".join(s) for t, s in q["context"]}
            q["support"] = {t + "\n" + ctx[t] for t in {t for t, _ in q["supporting_facts"]} if t in ctx}
            q["group"] = q["type"]
    DATA[name] = {"questions": qs[:N_Q], "corpus": corpus}
    print(f"{name}: {len(qs):,} questions, {len(corpus):,} passages | groups:",
          dict(Counter(q["group"] for q in qs)))
print("MODE =", MODE)

musique: 1,000 questions, 11,656 passages | groups: {'2-hop': 518, '3-hop': 316, '4-hop': 166}
2wikimultihopqa: 1,000 questions, 6,119 passages | groups: {'compositional': 413, 'comparison': 244, 'bridge_comparison': 235, 'inference': 108}
MODE = test


## The passage pools, embeddings and graph of Notebook 11 (read from its cache)

The next four cells are Notebook 11's code, unchanged, so that every embedding and extraction request matches its cache and nothing is recomputed.

In [2]:
for name, d in DATA.items():
    passages = [p["title"] + "\n" + p["text"] for p in d["corpus"]]
    if TEST:
        keep = set()
        for q in d["questions"]:
            if name == "musique":
                keep |= {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"]}
            else:
                keep |= {t + "\n" + " ".join(s) for t, s in q["context"]}
        passages = [p for p in passages if p in keep]
    d["passages"] = passages
    d["pid"] = {p: i for i, p in enumerate(passages)}
    found = np.mean([len(q["support"] & set(d["pid"])) / max(len(q["support"]), 1) for q in d["questions"]])
    print(f"{name}: {len(passages):,} passages in the pool; gold supporting paragraphs found in the pool: {found:.1%}")

musique: 484 passages in the pool; gold supporting paragraphs found in the pool: 100.0%
2wikimultihopqa: 229 passages in the pool; gold supporting paragraphs found in the pool: 100.0%


In [3]:
for name, d in DATA.items():
    d["P"] = llm.embed(f"{name}_passages", d["passages"])
    d["Q"] = llm.embed(f"{name}_questions", [q["question"] for q in d["questions"]])
    print(f"{name}: passage matrix {d['P'].shape}, question matrix {d['Q'].shape}")

musique: passage matrix (484, 1536), question matrix (25, 1536)
2wikimultihopqa: passage matrix (229, 1536), question matrix (25, 1536)


In [4]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str

class OpenIE(BaseModel):
    entities: list[str] = Field(description="named entities, dates and numbers mentioned in the passage")
    triples: list[Triple] = Field(description="factual subject-predicate-object triples stated in the passage")

OPENIE_SYS = ("Extract an open knowledge graph from a Wikipedia passage. List the named entities (people, places, "
              "organisations, works, events, dates, numbers) and every factual triple the passage states. Use full, "
              "canonical entity names (resolve pronouns to the entity they refer to); keep dates and numbers exact. At most "
              "40 triples, most important first.")

TRIPLE_RE = re.compile(r'\{\s*"subject":\s*"((?:[^"\\]|\\.)*)",\s*"predicate":\s*"((?:[^"\\]|\\.)*)",\s*"object":\s*"((?:[^"\\]|\\.)*)"\s*\}')

def salvage(text):
    # An output cut off at the token limit still contains complete triples: keep those.
    return {"entities": [], "triples": [{"subject": a, "predicate": b, "object": c} for a, b, c in TRIPLE_RE.findall(text)]}

for name, d in DATA.items():
    reqs = {f"p{i}": LLM.body([{"role": "system", "content": OPENIE_SYS}, {"role": "user", "content": p}],
                              max_tokens=4000, schema=OpenIE)
            for i, p in enumerate(d["passages"])}
    res = llm.run(f"{name}_openie", reqs, parse=OpenIE)
    d["openie"] = [res[f"p{i}"]["parsed"] or salvage(res[f"p{i}"].get("text", "")) for i in range(len(d["passages"]))]
    bad = sum(res[f"p{i}"]["parsed"] is None for i in range(len(d["passages"])))
    nt = sum(len(x["triples"]) for x in d["openie"])
    print(f"{name}: {len(reqs):,} extraction requests, {bad} cut off (complete triples kept), {nt:,} triples "
          f"({nt / len(reqs):.1f} per passage)")

musique: 484 extraction requests, 0 cut off (complete triples kept), 4,944 triples (10.2 per passage)


2wikimultihopqa: 229 extraction requests, 0 cut off (complete triples kept), 2,204 triples (9.6 per passage)


In [5]:
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()

def build_graph(d):
    # Nodes: passages 0..P-1, then entities. Edges: entity-entity (triple), entity-passage (mention), synonyms.
    n_p = len(d["passages"])
    ent_id, ent_name = {}, []
    def eid(s):
        k = norm(s)
        if not k:
            return None
        if k not in ent_id:
            ent_id[k] = n_p + len(ent_name); ent_name.append(s)
        return ent_id[k]
    rows, cols, facts = [], [], []
    for pi, ex in enumerate(d["openie"]):
        for e in ex["entities"]:
            j = eid(e)
            if j is not None:
                rows += [pi, j]; cols += [j, pi]
        for t in ex["triples"]:
            a, b = eid(t["subject"]), eid(t["object"])
            if a is None or b is None or a == b:
                continue
            rows += [a, b, pi, a, pi, b]; cols += [b, a, a, pi, b, pi]
            facts.append((a, b, pi, f"{t['subject']} {t['predicate']} {t['object']}"))
    d["n_p"], d["ent_name"], d["facts"] = n_p, ent_name, facts
    # Synonym edges between near-duplicate entity names (HippoRAG 2), computed on entity-name embeddings.
    E = llm.embed(f"{d['name']}_entities", ent_name, cap=200)
    syn = 0
    for s in range(0, len(ent_name), 512):
        S = E[s:s + 512] @ E.T
        ii, jj = np.where(S >= 0.9)
        for a, b in zip(ii + s, jj):
            if a < b:
                rows += [n_p + a, n_p + b]; cols += [n_p + b, n_p + a]; syn += 1
    n = n_p + len(ent_name)
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n))
    A.data[:] = 1.0                                  # duplicate edges count once
    deg = np.asarray(A.sum(1)).ravel(); deg[deg == 0] = 1
    d["T"] = (sp.diags(1 / deg) @ A).T.tocsr()        # column-stochastic transition matrix
    d["F"] = llm.embed(f"{d['name']}_facts", [f[3] for f in facts], cap=300)
    return n, syn

for name, d in DATA.items():
    d["name"] = name
    n, syn = build_graph(d)
    print(f"{name}: graph with {d['n_p']:,} passage + {len(d['ent_name']):,} entity nodes, "
          f"{len(d['facts']):,} triples, {syn:,} synonym edges")

musique: graph with 484 passage + 7,197 entity nodes, 4,936 triples, 356 synonym edges
2wikimultihopqa: graph with 229 passage + 2,894 entity nodes, 2,202 triples, 123 synonym edges


In [6]:
K = 5

def ppr(T, pers, alpha=0.5, iters=40):
    p = pers / pers.sum(); x = p.copy()
    for _ in range(iters):
        x = (1 - alpha) * p + alpha * (T @ x)
    return x

def retrieve(d, qi):
    q = d["Q"][qi]
    dense = d["P"] @ q
    vec = list(np.argsort(-dense)[:K])
    pers = np.zeros(d["T"].shape[0], np.float64)
    fs = d["F"] @ q if len(d["facts"]) else np.zeros(0)
    for j in np.argsort(-fs)[:5]:
        a, b, pi, _ = d["facts"][j]
        pers[a] += fs[j]; pers[b] += fs[j]
    for j in np.argsort(-dense)[:20]:
        pers[j] += 0.05 * max(dense[j], 0)
    if pers.sum() == 0:
        return vec, vec
    x = ppr(d["T"], pers)[:d["n_p"]]
    return vec, list(np.argsort(-x)[:K])

def recall(d, q, ids):
    if not q["support"]:
        return float("nan")
    got = {d["passages"][i] for i in ids}
    return len(q["support"] & got) / len(q["support"])

t0 = time.time()
for name, d in DATA.items():
    d["ret"] = [retrieve(d, i) for i in range(len(d["questions"]))]
    rv = np.nanmean([recall(d, q, v) for q, (v, g) in zip(d["questions"], d["ret"])])
    rg = np.nanmean([recall(d, q, g) for q, (v, g) in zip(d["questions"], d["ret"])])
    print(f"{name}: recall@5 of gold supporting paragraphs  vector {rv:.1%}  |  graph {rg:.1%}")
print(f"retrieval for {sum(len(d['questions']) for d in DATA.values()):,} questions in {time.time() - t0:.0f} s")

musique: recall@5 of gold supporting paragraphs  vector 61.0%  |  graph 69.3%
2wikimultihopqa: recall@5 of gold supporting paragraphs  vector 79.0%  |  graph 84.0%
retrieval for 50 questions in 0 s


## Condition (c): the facts most similar to the question, with their source passages

Each triple in `d["facts"]` is `(subject id, object id, passage id, text)`: the passage id is its provenance. The 15
triples most similar to the question are shown to the answerer; a citation of a triple resolves to its source passage.

In [7]:
N_FACTS = 15

def top_facts(d, qi):
    fs = d["F"] @ d["Q"][qi]
    return list(np.argsort(-fs)[:N_FACTS])

for name, d in DATA.items():
    d["fact_ids"] = [top_facts(d, i) for i in range(len(d["questions"]))]
    src_recall = np.nanmean([recall(d, q, {d["facts"][j][2] for j in fids})
                             for q, fids in zip(d["questions"], d["fact_ids"])])
    print(f"{name}: {N_FACTS} facts per question from on average "
          f"{np.mean([len({d['facts'][j][2] for j in f}) for f in d['fact_ids']]):.1f} source passages; "
          f"gold supporting paragraphs among those sources: {src_recall:.1%}")

musique: 15 facts per question from on average 5.9 source passages; gold supporting paragraphs among those sources: 64.7%
2wikimultihopqa: 15 facts per question from on average 3.8 source passages; gold supporting paragraphs among those sources: 76.0%


## Answers with citations (one Batch job per benchmark)

The same system prompt for all three conditions. Sources are numbered `[1]`, `[2]`, …; for condition (c) each source is
one fact. `sources()` returns, for each citation number, the passage it resolves to.

In [8]:
def final(text):
    m = re.findall(r"Answer:\s*(.+)", text)
    return (m[-1] if m else text.strip().splitlines()[-1] if text.strip() else "").strip()

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g); same = sum(common.values())
    if same == 0:
        return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

CITE_SYS = ("Answer the question using only the numbered sources. Reason in at most three short sentences and end "
            "each sentence with the numbers of the sources that support it in square brackets, for example [2] or "
            "[1][3]. Then give the final answer on the last line as 'Answer: <answer> [n]', citing the sources that "
            "support it. The answer must be a short phrase (an entity, date, number or yes/no), not a sentence.")
METHODS = ("vector", "graph", "facts")

def sources(d, qi, m):
    # Citation number -> (text shown to the answerer, passage id it resolves to).
    v, g = d["ret"][qi]
    if m == "facts":
        return {n + 1: (d["facts"][j][3], d["facts"][j][2]) for n, j in enumerate(d["fact_ids"][qi])}
    return {n + 1: (d["passages"][i], int(i)) for n, i in enumerate(v if m == "vector" else g)}

def prompt(d, qi, m):
    src = sources(d, qi, m)
    head = "Sources (facts extracted from Wikipedia passages):" if m == "facts" else "Sources:"
    body = "\n\n" if m != "facts" else "\n"
    return f"{head}\n" + body.join(f"[{n}] {t}" for n, (t, _) in src.items()) + \
           f"\n\nQuestion: {d['questions'][qi]['question']}"

for name, d in DATA.items():
    reqs = {f"{m}|{q['qid']}": LLM.body([{"role": "system", "content": CITE_SYS},
                                         {"role": "user", "content": prompt(d, qi, m)}], max_tokens=300)
            for qi, q in enumerate(d["questions"]) for m in METHODS}
    d["cited"] = llm.run(f"{name}_cited_answers", reqs)
    print(f"{name}: {len(reqs):,} cited answers")

musique: 75 cited answers
2wikimultihopqa: 75 cited answers


## Splitting an answer into statements and citations

A citation written after the full stop (`… in 1982. [2]`) is moved before it, the reasoning is split into sentences,
and each sentence keeps the numbers in its brackets. The answer line (or, if the model wrote none, the last line, as
in Notebook 11) becomes the statement *“The answer to ⟨question⟩
is ⟨answer⟩.”* A number outside the sources shown counts as an invalid citation.

In [9]:
CIT = re.compile(r"\[(\d+(?:\s*,\s*\d+)*)\]")

def nums(s):
    return [int(x) for g in CIT.findall(s) for x in re.split(r"\s*,\s*", g)]

def statements(text, question):
    # -> [(statement without citations, [citation numbers])], the answer statement last
    lines = text.strip().splitlines()
    ai = max((i for i, l in enumerate(lines) if l.strip().startswith("Answer:")), default=None)
    if ai is None and lines:
        ai = max(i for i, l in enumerate(lines) if l.strip())   # no 'Answer:' line: the last line is the answer
    reasoning = " ".join(lines[:ai] if ai is not None else lines)
    reasoning = re.sub(r"\.\s*((?:\s*\[\d+(?:\s*,\s*\d+)*\])+)", r" \1.", reasoning)
    out = []
    for s in re.split(r"(?<=[.!?])\s+", reasoning.strip()):
        clean = re.sub(r"\s+", " ", CIT.sub("", s)).strip(" .")
        if clean:
            out.append((clean + ".", nums(s)))
    if ai is not None:
        a = lines[ai].split("Answer:", 1)[-1]
        ans = re.sub(r"\s+", " ", CIT.sub("", a)).strip(" .")
        out.append((f"The answer to “{question.rstrip('?')}?” is {ans}.", nums(a)))
    return out

for name, d in DATA.items():
    d["parsed"] = {}
    for qi, q in enumerate(d["questions"]):
        for m in METHODS:
            src = sources(d, qi, m)
            st = statements(d["cited"][f"{m}|{q['qid']}"]["text"], q["question"])
            # resolve every citation to a passage id; invalid numbers are kept as None
            d["parsed"][(qi, m)] = [(s, [src[n][1] if n in src else None for n in c]) for s, c in st]
    n_st = np.mean([len(v) for v in d["parsed"].values()])
    uncited = np.mean([not c for v in d["parsed"].values() for _, c in v])
    invalid = sum(p is None for v in d["parsed"].values() for _, c in v for p in c)
    print(f"{name}: {n_st:.1f} statements per answer, {uncited:.1%} of statements without a citation, "
          f"{invalid} invalid citation numbers")

d = DATA["musique"]
print("\nExample:", d["questions"][0]["question"])
for m in METHODS:
    print(f"\n{m.upper()}: {d['cited'][f'{m}|{d['questions'][0]['qid']}']['text']!r}")
    for s, c in d["parsed"][(0, m)]:
        print(f"   {s!r}  ->  passages {c}")

musique: 1.7 statements per answer, 0.8% of statements without a citation, 0 invalid citation numbers
2wikimultihopqa: 1.6 statements per answer, 0.8% of statements without a citation, 0 invalid citation numbers

Example: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?

VECTOR: 'Answer: June 1982 [3]'
   "The answer to “When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?” is June 1982."  ->  passages [2]

GRAPH: 'Diego Maradona was signed by Barcelona in June 1982. [2]\n\nAnswer: June 1982 [2]'
   'Diego Maradona was signed by Barcelona in June 1982.'  ->  passages [2]
   "The answer to “When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?” is June 1982."  ->  passages [2]

FACTS: 'Diego Maradona was signed by Barcelona in June 1982. [4][10]\n\nAnswer: June 1982 [4]'
   'Diego Maradona was signed by Barcelona in June 1982.'  ->  passages [2, 1]
   "The answer to “When

## The ALCE-style citation judge (one structured call per statement)

As in ALCE, every statement is checked only against the passages **it** cites (labelled `P1`, `P2`, … within the call),
so a passage cited elsewhere in the answer cannot lend it support. The judge decides whether the cited passages
**together entail** the statement (citation recall) and which of them are **relevant** to it (citation precision).
Statements without a valid citation are not sent: they count as unsupported. Abstentions (“not stated in the
sources”, “cannot be determined”) are not claims and are not sent either: they count as unsupported, their citations as
not relevant, and the share of abstained answers is reported separately.

In [10]:
class Verdict(BaseModel):
    supported: bool = Field(description="the passages together fully entail the statement")
    relevant: list[str] = Field(description="labels of the passages that contain information needed for the statement")

JUDGE_SYS = ("You check one citation of an answer, as in the ALCE benchmark. You receive a statement and the passages "
             "it cites. 'supported' is true only if the passages together fully entail the statement: every fact in "
             "it is stated in, or directly implied by, them; your own knowledge does not count. 'relevant' lists the "
             "passages that contain information needed for the statement; a passage that is off-topic or adds "
             "nothing needed is not relevant.")

def judge_body(d, statement, pids):
    msg = ("Passages:\n" + "\n\n".join(f"[P{n + 1}] {d['passages'][p]}" for n, p in enumerate(pids)) +
           f"\n\nStatement: {statement}")
    return LLM.body([{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": msg}],
                    max_tokens=200, schema=Verdict)

REFUSAL = re.compile(r"not (stated|given|mentioned|specified|provided|available|determinable)|cannot be (determined|answered)|"
                     r"(do|does) not (say|state|give|mention|specify)|no information|insufficient|^unknown\b", re.I)

def refusal(statement):
    # "The sources do not say ..." / "The answer ... is Not stated." is an abstention, not a claim to cite
    return bool(REFUSAL.search(statement))

def cited_pids(c):
    return list(dict.fromkeys(p for p in c if p is not None))   # valid citations, first-cited order

for name, d in DATA.items():
    reqs = {}
    for qi, q in enumerate(d["questions"]):
        for m in METHODS:
            for k, (st, c) in enumerate(d["parsed"][(qi, m)]):
                if cited_pids(c) and not refusal(st):
                    reqs[f"{m}|{q['qid']}|{k}"] = judge_body(d, st, cited_pids(c))
    d["judged"] = llm.run(f"{name}_citation_judge", reqs, parse=Verdict)
    bad = sum(r["parsed"] is None for r in d["judged"].values())
    print(f"{name}: {len(reqs):,} judge calls, one per cited statement ({bad} unparsed)")

musique: 95 judge calls, one per cited statement (0 unparsed)
2wikimultihopqa: 81 judge calls, one per cited statement (0 unparsed)


## Scores per answer

- **EM, F1:** answer quality, as in Notebook 11.
- **Gold precision / recall:** cited passages against the gold supporting paragraphs; an answer without citations
  scores 0 on both.
- **ALCE recall:** the share of statements judged supported (uncited or invalid-only statements count as
  unsupported). **ALCE precision:** the share of (statement, valid cited passage) pairs judged relevant; invalid
  citation numbers count as not relevant; an answer without citations scores 0.
- **Correct and supported:** the answer is correct (F1 ≥ 0.5) *and* its answer statement is supported by its
  citations. Citation recall alone rewards a well-cited refusal (“cannot be determined”); this measure does not.

In [11]:
def score(d, qi, m):
    q = d["questions"][qi]
    pred = final(d["cited"][f"{m}|{q['qid']}"]["text"])
    pred = re.sub(r"\s+", " ", CIT.sub("", pred)).strip(" .")
    parsed = d["parsed"][(qi, m)]
    cited = {p for _, c in parsed for p in c if p is not None}
    gold = {d["pid"][s] for s in q["support"] if s in d["pid"]}
    row = {"em": float(max(normalize(pred) == normalize(g) for g in q["golds"])),
           "f1": max(f1(pred, g) for g in q["golds"]),
           "g_prec": len(cited & gold) / len(cited) if cited else 0.0,
           "g_rec": len(cited & gold) / len(gold) if gold else float("nan"),
           "n_cit": sum(len(c) for _, c in parsed)}
    sup, rel, pairs, ans_sup = 0, 0, 0, False
    for k, (st, c) in enumerate(parsed):
        pids = cited_pids(c)
        pairs += len(pids) + sum(p is None for p in c)             # invalid numbers count as not relevant
        r = d["judged"].get(f"{m}|{q['qid']}|{k}") if pids and not refusal(st) else None
        v = r["parsed"] if r and r["parsed"] else {"supported": False, "relevant": []}
        sup += bool(v["supported"])
        rel += len({f"P{n + 1}" for n in range(len(pids))} & set(v["relevant"]))
        ans_sup = bool(v["supported"]) if st.startswith("The answer to") and k == len(parsed) - 1 else ans_sup
    row["sup_correct"] = float(ans_sup and row["f1"] >= 0.5)
    row["abstain"] = float(refusal(pred))
    row["a_rec"] = sup / len(parsed) if parsed else 0.0
    row["a_prec"] = rel / pairs if pairs else 0.0
    return row

for name, d in DATA.items():
    d["scores"] = [{"group": q["group"], **{m: score(d, qi, m) for m in METHODS}} for qi, q in enumerate(d["questions"])]
print("scored", sum(len(d["scores"]) for d in DATA.values()), "questions x", len(METHODS), "conditions")

scored 50 questions x 3 conditions


## Results with 95% confidence intervals

Means per condition; the last two columns are paired bootstrap intervals over questions (graph − vector and
facts − vector). An interval that excludes 0 is a difference this sample can resolve. Gold recall is averaged over
questions that have gold supporting paragraphs in the pool.

In [12]:
METRICS = [("f1", "answer F1"), ("em", "exact match"), ("g_prec", "gold citation precision"),
           ("g_rec", "gold citation recall"), ("a_rec", "ALCE citation recall"), ("a_prec", "ALCE citation precision"),
           ("sup_correct", "correct and supported"), ("abstain", "answer abstained")]
SUMMARY = {}

def table(label, rows):
    print(f"\n{label}  (n = {len(rows)})")
    print(f"{'metric':<26}{'vector':>8}{'graph':>8}{'facts':>8}{'graph - vector [95% CI]':>28}{'facts - vector [95% CI]':>28}")
    for k, lab in METRICS:
        ok = [r for r in rows if not np.isnan(r["vector"][k])]
        means = {m: np.mean([r[m][k] for r in ok]) for m in METHODS}
        diffs = {m: paired_boot([r["vector"][k] for r in ok], [r[m][k] for r in ok]) for m in ("graph", "facts")}
        SUMMARY[(label, k)] = (len(ok), means, diffs)
        ci = lambda t: f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]"
        print(f"{lab:<26}{means['vector']:>8.1%}{means['graph']:>8.1%}{means['facts']:>8.1%}"
              f"{ci(diffs['graph']):>28}{ci(diffs['facts']):>28}")

for name, d in DATA.items():
    table(name, d["scores"])
table("both benchmarks", [s for d in DATA.values() for s in d["scores"]])
print("\ncitations per answer:", ", ".join(
    f"{m} {np.mean([s[m]['n_cit'] for d in DATA.values() for s in d['scores']]):.1f}" for m in METHODS))


musique  (n = 25)
metric                      vector   graph   facts     graph - vector [95% CI]     facts - vector [95% CI]
answer F1                    46.0%   55.3%   48.2%       +9.3% [-6.7%, +25.3%]      +2.2% [-13.8%, +18.7%]
exact match                  40.0%   48.0%   36.0%       +8.0% [-8.0%, +24.0%]      -4.0% [-20.0%, +12.0%]
gold citation precision      80.4%   81.3%   77.7%      +0.9% [-14.0%, +15.7%]      -2.7% [-18.0%, +10.8%]
gold citation recall         48.7%   56.0%   50.3%       +7.3% [-4.3%, +20.7%]      +1.7% [-10.7%, +13.3%]
ALCE citation recall         28.0%   30.7%   31.3%      +2.7% [-14.0%, +18.7%]      +3.3% [-11.3%, +18.0%]
ALCE citation precision      55.0%   62.3%   62.0%      +7.3% [-11.7%, +26.0%]      +7.0% [-11.3%, +25.3%]
correct and supported        16.0%   16.0%   12.0%      +0.0% [-12.0%, +12.0%]       -4.0% [-16.0%, +8.0%]
answer abstained             32.0%   20.0%   24.0%      -12.0% [-32.0%, +8.0%]      -8.0% [-28.0%, +12.0%]

2wikimultihopqa  

### One question, three cited answers

The first 2Wiki question on which the facts condition scores a higher citation recall than vector RAG (an illustration, not a typical case): the judge's verdict for every statement, with the cited passages marked `*` when they are gold supporting paragraphs.

In [13]:
d = DATA["2wikimultihopqa"]
idx = next((i for i, s in enumerate(d["scores"]) if s["facts"]["a_rec"] > s["vector"]["a_rec"]
            and not s["facts"]["abstain"]), 0)
q = d["questions"][idx]
gold = {d["pid"][s] for s in q["support"] if s in d["pid"]}
print(f"Q: {q['question']}\nGold: {q['golds'][0]}")
for m in METHODS:
    print(f"\n{m.upper()}  (ALCE recall {d['scores'][idx][m]['a_rec']:.0%}, precision {d['scores'][idx][m]['a_prec']:.0%})")
    for k, (st, c) in enumerate(d["parsed"][(idx, m)]):
        pids = cited_pids(c)
        r = d["judged"].get(f"{m}|{q['qid']}|{k}") if pids and not refusal(st) else None
        v = r["parsed"] if r and r["parsed"] else {}
        print(f"  {'supported' if v.get('supported') else 'NOT supported':<14} {st}")
        for n, p in enumerate(pids):
            rel = "relevant" if f"P{n + 1}" in v.get("relevant", []) else "not relevant"
            print(f"      {'*' if p in gold else ' '} P{n + 1} ({rel}): {d['passages'][p][:90].replace(chr(10), ' | ')}")

Q: Who is Raghnall Mac Ruaidhrí's paternal grandfather?
Gold: Ailéan mac Ruaidhrí

VECTOR  (ALCE recall 0%, precision 0%)
  NOT supported  The answer to “Who is Raghnall Mac Ruaidhrí's paternal grandfather?” is Ailéan mac Ruaidhrí.
      * P1 (not relevant): Ruaidhrí Mac Ruaidhrí | Ruaidhrí Mac Ruaidhrí (died 14 October 1318?) was a fourteenth-centu

GRAPH  (ALCE recall 0%, precision 0%)
  NOT supported  The answer to “Who is Raghnall Mac Ruaidhrí's paternal grandfather?” is Ailéan mac Ruaidhrí.
      * P1 (not relevant): Ruaidhrí Mac Ruaidhrí | Ruaidhrí Mac Ruaidhrí (died 14 October 1318?) was a fourteenth-centu

FACTS  (ALCE recall 100%, precision 100%)
  supported      The answer to “Who is Raghnall Mac Ruaidhrí's paternal grandfather?” is Ailéan mac Ruaidhri.
      * P1 (relevant): Raghnall Mac Ruaidhrí | Raghnall Mac Ruaidhrí (died October 1346) was an eminent Scottish ma
      * P2 (relevant): Ruaidhrí Mac Ruaidhrí | Ruaidhrí Mac Ruaidhrí (died 14 October 1318?) was a fourteenth-

## What this test cost

Only the two new stages are paid here; the passage embeddings, the extraction and the graph are reused from Notebook 11 (its cost: about $2.21 for the graph index).

In [14]:
STAGES = [f"{n}_{s}" for n in DATA for s in ("cited_answers", "citation_judge")]
LEDGER = llm.ledger(STAGES)
print(f"{'stage':<36}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for s in STAGES:
    a = LEDGER[s]
    print(f"{s:<36}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.2f}{a['batch_share']:>11.0%}")
print(f"{'TOTAL (new calls only)':<36}{sum(LEDGER[s]['calls'] for s in STAGES):>8,}"
      f"{sum(LEDGER[s]['in'] for s in STAGES):>13,}{sum(LEDGER[s]['out'] for s in STAGES):>12,}"
      f"{sum(LEDGER[s]['usd'] for s in STAGES):>8.2f}")

stage                                  calls    input tok  output tok     USD  via Batch
musique_cited_answers                     75       45,485       2,024    0.01         0%
musique_citation_judge                   123       46,121       2,319    0.01         0%
2wikimultihopqa_cited_answers             75       52,565       1,922    0.01         0%
2wikimultihopqa_citation_judge           121       53,985       2,374    0.01         0%
TOTAL (new calls only)                   394      198,156       8,639    0.02


## Limitations

- **Judge.** ALCE scores support with an NLI model (TRUE, T5-11B); here an LLM judge (`gpt-6-luna`) labels each statement in
  its own call. The judge should be validated on a sample of **CAQA** (Hu et al., ACL 2025), a
  benchmark built from knowledge graphs that labels citations as supportive, partial, contradictory or irrelevant,
  and re-scored with a second judge as in `second_judge.py`.
- **Strict answer statement.** The answer statement contains the whole question, so for a multi-hop question it is
  supported only if the answer cites the passages of every hop. This is ALCE's standard, but it makes “correct and
  supported” much lower than answer accuracy.
- **Unequal text.** Condition (c) sees 15 one-line facts, much less text than five passages; a lower answer F1 there
  reflects the context, not the citations. Its citations are judged against the source passages, which the answerer
  did not see.
- **Gold labels.** The gold supporting paragraphs mark the evidence the benchmark intended; a passage outside them can
  still support a statement, so gold precision is a lower bound and the ALCE measures are the primary result.
- **One configuration.** One model, one prompt, top-5 passages and 15 facts; each configuration was run once.

**References.** Gao et al., *Enabling Large Language Models to Generate Text with Citations* (ALCE), EMNLP 2023 ·
Li et al., *Towards Verifiable Generation: A Benchmark for Knowledge-aware Language Model Attribution* (KaLMA),
Findings of ACL 2024 · Hu et al., *Can LLMs Evaluate Complex Attribution in QA? Automatic Benchmarking Using Knowledge
Graphs* (CAQA), ACL 2025.

In [15]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 20: citation accuracy, {MODE} run, gpt-6-luna (no reasoning), MuSiQue + 2Wiki")
print("=" * 120)
print(f"{'benchmark':<18}{'metric':<26}{'vector':>8}{'graph':>8}{'facts':>8}{'graph - vector [95% CI]':>28}{'facts - vector [95% CI]':>28}")
verdicts = {}
for label in list(DATA) + ["both benchmarks"]:
    for k, lab in METRICS:
        if k in ("em", "abstain"):
            continue
        n, means, diffs = SUMMARY[(label, k)]
        g, f = diffs["graph"], diffs["facts"]
        print(f"{label:<18}{lab:<26}{means['vector']:>8.1%}{means['graph']:>8.1%}{means['facts']:>8.1%}"
              f"{f'{g[0]:+.1%} [{g[1]:+.1%}, {g[2]:+.1%}]':>28}{f'{f[0]:+.1%} [{f[1]:+.1%}, {f[2]:+.1%}]':>28}")
        if label == "both benchmarks" and k in ("a_rec", "a_prec", "sup_correct"):
            for m, t in (("graph", g), ("facts", f)):
                verdicts[f"{m} {lab}"] = "better" if t[1] > 0 else "worse" if t[2] < 0 else "no resolvable difference"
print("Answers that abstained (both benchmarks):", ", ".join(
    f"{m} {SUMMARY[('both benchmarks', 'abstain')][1][m]:.1%}" for m in METHODS))
print("\nVerdict versus vector RAG (both benchmarks):", "; ".join(f"{k}: {v}" for k, v in verdicts.items()))
print("\nLecture message (Slide 5, Task 3.2): provenance lets a graph point to the passage behind each fact. Whether"
      "\nthat makes an agent's citations more accurate than citing retrieved passages is what the verdict line says.")

RESULTS SUMMARY — Notebook 20: citation accuracy, test run, gpt-6-luna (no reasoning), MuSiQue + 2Wiki
benchmark         metric                      vector   graph   facts     graph - vector [95% CI]     facts - vector [95% CI]
musique           answer F1                    46.0%   55.3%   48.2%       +9.3% [-6.7%, +25.3%]      +2.2% [-13.8%, +18.7%]
musique           gold citation precision      80.4%   81.3%   77.7%      +0.9% [-14.0%, +15.7%]      -2.7% [-18.0%, +10.8%]
musique           gold citation recall         48.7%   56.0%   50.3%       +7.3% [-4.3%, +20.7%]      +1.7% [-10.7%, +13.3%]
musique           ALCE citation recall         28.0%   30.7%   31.3%      +2.7% [-14.0%, +18.7%]      +3.3% [-11.3%, +18.0%]
musique           ALCE citation precision      55.0%   62.3%   62.0%      +7.3% [-11.7%, +26.0%]      +7.0% [-11.3%, +25.3%]
musique           correct and supported        16.0%   16.0%   12.0%      +0.0% [-12.0%, +12.0%]       -4.0% [-16.0%, +8.0%]
2wikimultihopqa   answ